# Part 1 — Python & Data Foundations

CareLoop Clinics — No-Show Guard


In [ ]:
import csv
import json
import csv
import json


def parse_row(row):
    # Copy the row so a failed conversion doesn't change the raw record.
    parsed = row.copy()
    parsed["age"] = int(row["age"])
    parsed["fee_inr"] = int(row["fee_inr"])
    parsed["prior_no_shows"] = int(row["prior_no_shows"])
    parsed["distance_km"] = float(row["distance_km"])
    return parsed


accepted = []
rejected = []
row_count = 0

with open("appointments.csv", newline="", encoding="utf-8-sig") as file:
    reader = csv.DictReader(file)
    print("Header:", reader.fieldnames)

    for row in reader:
        row_count += 1

        try:
            parsed_row = parse_row(row)
        except ValueError:
            rejected.append(row)
        else:
            accepted.append(parsed_row)

with open("rejected_rows.json", "w", encoding="utf-8") as file:
    json.dump(rejected, file, indent=2)

print("Rows read:", row_count)
print("Accepted:", len(accepted))
print("Rejected:", len(rejected))

Header: ['appointment_id', 'patient_id', 'clinic_id', 'booked_on', 'appt_date', 'age', 'gender', 'distance_km', 'fee_inr', 'booking_channel', 'visit_type', 'reminder_sent', 'prior_no_shows', 'status']
Rows read: 1020
Accepted: 988
Rejected: 32


#Task 1.2

In [ ]:
import pandas as pd

df = pd.read_csv("appointments.csv")
display(df.head())

print("Shape:", df.shape)
print("\nColumn types:")
df.info()
print("\nMissing values:\n", df.isnull().sum())
print("\nExact duplicates:", df.duplicated().sum())

df = df.drop_duplicates().copy()
print("Rows after removing duplicates:", len(df))

df["booking_channel"] = df["booking_channel"].str.strip().str.title()
print("Rows after standardizing booking channels:", len(df))
print("Booking-channel labels:", df["booking_channel"].unique())

df["appt_date"] = pd.to_datetime(df["appt_date"], errors="coerce")
df["booked_on"] = pd.to_datetime(df["booked_on"])
print("Rows after converting dates:", len(df))
df = df.dropna(subset=["appt_date", "status"]).copy()
print("Rows after dropping invalid appointment dates or missing statuses:", len(df))

,appointment_id,patient_id,clinic_id,booked_on,appt_date,age,gender,distance_km,fee_inr,booking_channel,visit_type,reminder_sent,prior_no_shows,status
0,A0001,P509,C05,2025-01-24,2025-01-25,60.0,F,18.0,850,Phone,Consultation,Yes,1,No-Show
1,A0002,P046,C07,2025-05-27,2025-06-06,39.0,M,12.5,350,Walk-in,Follow-up,No,2,Attended
2,A0003,P009,C02,2025-03-24,2025-03-30,63.0,F,4.1,850,Walk-in,Consultation,No,0,Attended
3,A0003,P009,C02,2025-03-24,2025-03-30,63.0,F,4.1,850,Walk-in,Consultation,No,0,Attended
4,A0004,P482,C01,2025-04-18,2025-04-22,52.0,F,7.5,750,App,Consultation,Yes,0,Attended


Shape: (1020, 14)

Column types:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1020 entries, 0 to 1019
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   appointment_id   1020 non-null   object 
 1   patient_id       1020 non-null   object 
 2   clinic_id        1020 non-null   object 
 3   booked_on        1020 non-null   object 
 4   appt_date        1019 non-null   object 
 5   age              1008 non-null   float64
 6   gender           1020 non-null   object 
 7   distance_km      1000 non-null   float64
 8   fee_inr          1020 non-null   int64  
 9   booking_channel  1020 non-null   object 
 10  visit_type       1020 non-null   object 
 11  reminder_sent    1020 non-null   object 
 12  prior_no_shows   1020 non-null   int64  
 13  status           1016 non-null   object 
dtypes: float64(2), int64(2), object(10)
memory usage: 111.7+ KB

Missing values:
 appointment_id      0
patient_id         

#Task 1.3

In [ ]:
import numpy as np

# (a) Mark invalid ages as missing.
invalid_age = df["age"].lt(1) | df["age"].gt(100)
df.loc[invalid_age, "age"] = np.nan

# (b) Fill missing values using each column's median.
df["age"] = df["age"].fillna(df["age"].median()).round().astype(int)
df["distance_km"] = df["distance_km"].fillna(df["distance_km"].median())

# (c) Calculate the fences after filling gaps, then cap outliers.
for col in ["distance_km", "fee_inr"]:
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1

    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr

    capped_count = ((df[col] < lower) | (df[col] > upper)).sum()
    df[col] = np.clip(df[col], lower, upper)
    print(f"{col}: {capped_count} values capped")

# (d) Round after clipping.
df["distance_km"] = df["distance_km"].round(1)
df["fee_inr"] = df["fee_inr"].round().astype(int)

distance_km: 26 values capped
fee_inr: 3 values capped


#Task 1.4

In [ ]:
import numpy as np

# Create the two features.
df["lead_days"] = (df["appt_date"] - df["booked_on"]).dt.days
df["no_show"] = df["status"].eq("No-Show").astype(int)

# Build the NumPy array in the required column order.
columns = ["age", "distance_km", "fee_inr", "lead_days"]
X = df[columns].to_numpy()

# (a) Mean of each column.
print("Column order:", columns)
print("Column means:", X.mean(axis=0).round(2))

# (b) Count and share of appointments booked more than 14 days ahead.
mask = X[:, 3] > 14
print("Count with lead_days > 14:", mask.sum())
print(f"Share with lead_days > 14: {mask.mean():.1%}")

# (c) Distance z-scores and count outside ±2.
x = X[:, 1]
z_scores = (x - np.mean(x)) / np.std(x)
print("Count with |distance z-score| > 2:", (np.abs(z_scores) > 2).sum())

Column order: ['age', 'distance_km', 'fee_inr', 'lead_days']
Column means: [ 48.17   9.5  635.71  11.82]
Count with lead_days > 14: 296
Share with lead_days > 14: 29.9%
Count with |distance z-score| > 2: 53


#Task 1.5

In [ ]:
import pandas as pd
import requests

clinics = pd.read_csv("clinics.csv")
url = "https://api.open-meteo.com/v1/forecast"


def get_temperature(clinic):
    clinic_name = clinic.get("clinic_name", clinic.get("name", clinic["clinic_id"]))
    params = {
        "latitude": clinic["latitude"],
        "longitude": clinic["longitude"],
        "hourly": "temperature_2m",
        "forecast_days": 1,
        "timezone": "UTC",
    }

    for attempt in range(2):  # First attempt plus one retry
        try:
            response = requests.get(url, params=params, timeout=10)

            if response.status_code != 200:
                reason = f"HTTP {response.status_code}"
                continue

            data = response.json()
            temperature = data["hourly"]["temperature_2m"][0]

            if temperature is None:
                reason = "first hourly temperature is missing"
                continue

            return temperature

        except requests.exceptions.RequestException as exc:
            reason = str(exc)
        except (KeyError, IndexError, TypeError):
            reason = "hourly temperature data is missing or malformed"

    print(f"{clinic_name}: {reason}")
    return None


clinics["temp_c"] = [get_temperature(clinic) for _, clinic in clinics.iterrows()]
clinics.to_csv("clinics_enriched.csv", index=False)

print("Rows saved:", len(clinics))
print("Missing temperatures:", clinics["temp_c"].isna().sum())

Rows saved: 8
Missing temperatures: 0


#Task 1.6

In [ ]:
from numbers import Number

import pandas as pd

appointment_columns = [
    "appointment_id", "patient_id", "clinic_id", "booked_on", "appt_date",
    "age", "gender", "distance_km", "fee_inr", "booking_channel",
    "visit_type", "reminder_sent", "prior_no_shows", "status",
    "lead_days", "no_show",
]

# Validate before exporting.
appointments_clean = (
    df.loc[:, appointment_columns]
      .sort_values("appointment_id")
      .copy()
)

ids_are_unique = appointments_clean["appointment_id"].is_unique
missing_count = appointments_clean.isna().sum().sum()

print("Appointment IDs are unique:", ids_are_unique)
print("Missing values across the 16 columns:", missing_count)

if not ids_are_unique or missing_count != 0:
    raise ValueError("Fix duplicate appointment IDs or missing values before exporting.")

for col in ["booked_on", "appt_date"]:
    appointments_clean[col] = appointments_clean[col].dt.strftime("%Y-%m-%d")

appointments_clean.to_csv("appointments_clean.csv", index=False)
clinics_enriched = pd.read_csv("clinics_enriched.csv")


def sql_value(value):
    if pd.isna(value):
        return "NULL"
    if isinstance(value, Number):
        return str(value)
    return f"'{value}'"


insert_count = 0

with open("load_data.sql", "w", encoding="utf-8") as file:
    # Keep the clinic columns in the order used by the clinics table.
    for row in clinics_enriched.itertuples(index=False, name=None):
        values = ", ".join(sql_value(value) for value in row)
        file.write(f"INSERT INTO clinics VALUES ({values});\n")
        insert_count += 1

    for row in appointments_clean.itertuples(index=False, name=None):
        values = ", ".join(sql_value(value) for value in row)
        file.write(f"INSERT INTO appointments VALUES ({values});\n")
        insert_count += 1

print("Appointments CSV shape:", appointments_clean.shape)
print("Clinic rows:", len(clinics_enriched))
print("SQL INSERT statements:", insert_count)

Appointment IDs are unique: True
Missing values across the 16 columns: 0
Appointments CSV shape: (990, 16)
Clinic rows: 8
SQL INSERT statements: 998
